<a href="https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis and time window

One row represents one anonymized content page in the public-safe FlyRank starter dataset. The analysis is therefore conducted at the page level.

The starter file does not contain a reliable warehouse month or date-window field, so I do not claim a specific monthly observation window. Instead, I use the page-level snapshot supplied in the internship repository. All analysis in this notebook is restricted to this public-safe release.

In [18]:
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/Kaunaingul-ai/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

data_path = os.path.join(
    REPO_DIR,
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate content_id rows:", df["content_id"].duplicated().sum())
print("Unique content_id values:", df["content_id"].nunique())
print("One row per content page:", df["content_id"].nunique() == len(df))

Rows: 30000
Columns: 44
Duplicate content_id rows: 0
Unique content_id values: 30000
One row per content page: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field roles

For this capstone, I separate the available variables into four roles.

**Features used for modeling**
- `impressions_90d`
- `ctr`
- `avg_position`
- `content_age_days`
- `days_since_last_update`
- `word_count`

These variables describe page-level search visibility, engagement, position, and content characteristics available before model evaluation.

**Label**
- `trend_direction`

The observed label is whether `trend_direction` is `"down"`. It is used only as the evaluation target and is not included as a model input.

**Context / identifiers**
- `content_id`

This identifies each anonymized page but is not used as a predictive feature.

**Excluded**
Any fields that directly reveal the observed outcome, future information, private client information, raw URLs, domains, or query text are excluded from the modeling inputs. This helps reduce leakage and keeps the analysis public-safe.

In [19]:
feature_cols = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

label_col = "trend_direction"
context_cols = ["content_id"]

print("Features:")
for col in feature_cols:
    print("-", col, "| present:", col in df.columns)

print("\nLabel:")
print("-", label_col, "| present:", label_col in df.columns)

print("\nContext:")
for col in context_cols:
    print("-", col, "| present:", col in df.columns)

print("\nLabel used as feature:", label_col in feature_cols)

Features:
- impressions_90d | present: True
- ctr | present: True
- avg_position | present: True
- content_age_days | present: True
- days_since_last_update | present: True
- word_count | present: True

Label:
- trend_direction | present: True

Context:
- content_id | present: True

Label used as feature: False


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification checks

I verify the data contract with simple reproducible checks rather than relying only on assumptions. The checks below confirm the dataset grain, row count, missingness in the selected modeling fields, and whether a usable time-window field exists in the public-safe starter release.

Because the starter dataset does not contain a reliable warehouse month field, the analysis is treated as a page-level snapshot rather than a month-specific warehouse slice.

In [20]:
selected_cols = feature_cols + [label_col] + context_cols

print("Dataset grain checks")
print("--------------------")
print("Rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())
print("Duplicate content_id rows:", df["content_id"].duplicated().sum())

print("\nMissing values in selected fields")
print("---------------------------------")
print(df[selected_cols].isna().sum())

possible_time_cols = [
    c for c in df.columns
    if any(term in c.lower() for term in ["date", "month", "window", "period"])
]

print("\nPossible date/window fields")
print("---------------------------")
print(possible_time_cols if possible_time_cols else "None found")

print("\nLabel distribution")
print("------------------")
print(df[label_col].value_counts(dropna=False))

Dataset grain checks
--------------------
Rows: 30000
Unique content pages: 30000
Duplicate content_id rows: 0

Missing values in selected fields
---------------------------------
impressions_90d              0
ctr                          0
avg_position                 0
content_age_days             0
days_since_last_update       0
word_count                7699
trend_direction              0
content_id                   0
dtype: int64

Possible date/window fields
---------------------------
['days_since_last_update']

Label distribution
------------------
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This starter dataset is useful for building and validating a public-safe ranking workflow, but it has several limitations.

First, it is a static page-level snapshot rather than a full historical warehouse panel, so it cannot support strong claims about long-term temporal change or causal effects. The field `days_since_last_update` measures page recency, but it is not the date on which the observation was recorded.

Second, `word_count` contains missing values, so models using that feature need an explicit missing-value strategy rather than assuming complete data.

Third, the observed outcome is not perfectly balanced across classes. The `"down"` category is the largest group, so raw accuracy alone would not be an appropriate measure of ranking quality.

Finally, the starter release intentionally excludes private client information, raw URLs, domains, query text, and other contextual business information. Model recommendations should therefore be treated as decision support, not as automatic editorial decisions.

In [21]:

print("Data-limit checks")
print("-----------------")

print("word_count missing:", int(df["word_count"].isna().sum()))
print(
    "word_count missing rate:",
    round(df["word_count"].isna().mean(), 3)
)

label_share = (
    df["trend_direction"]
    .value_counts(normalize=True)
    .round(3)
)

print("\nLabel shares:")
print(label_share)

private_like_terms = [
    "url",
    "domain",
    "client",
    "query"
]

private_like_cols = [
    c for c in df.columns
    if any(term in c.lower() for term in private_like_terms)
]

print("\nColumns with private/context-like names:")
print(private_like_cols if private_like_cols else "None found")


Data-limit checks
-----------------
word_count missing: 7699
word_count missing rate: 0.257

Label shares:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64

Columns with private/context-like names:
['client_id']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.